In [20]:
import os
from llama_parse import LlamaParse
from langchain_core.documents import Document as LC_Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import faiss

In [2]:
from dotenv import load_dotenv

load_dotenv()
api_lp_key = os.getenv("LLAMA_CLOUD_API_KEY")

Load the Data and Parsing the data

In [3]:
parser = LlamaParse(
    api_key = api_lp_key,
    result_type = "markdown", # For simple documents you can text instead of markdown. Markdown is more reliable when reading the tables and images
    num_workers = 2,
    verbose = True,
    language = "en"
)

file_path = "./file_survey_paper.pdf"
documents = parser.load_data(file_path)


Started parsing the file under job_id d27598ac-7c9f-49b2-b208-40012564a0b0
...

In [4]:
print(f"Loaded {len(documents)} document pages/sections.")
print(f"Sample Content:\n{documents[0].text[:500]}...")

Loaded 144 document pages/sections.
Sample Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Yang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†

# Abstract

Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. L...


Using recursive splitting and Semantic chunking

In [5]:
embed_model = OllamaEmbeddings(model="nomic-embed-text")


# # convert LlamaIndex Documents -> LangChain Documents
# lc_documents = [
#     LC_Document(
#         page_content=doc.text,
#         metadata=doc.metadata
#     )
#     for doc in documents
# ]

# Convert LlamaIndex Documents → LangChain Documents with explicit metadata mapping
lc_documents = []
for doc in documents:
    # LlamaParse usually uses 'page_label' for the human-readable page number
    page_num = doc.metadata.get("page_label", "Unknown")
    
    lc_documents.append(
        LC_Document(
            page_content=doc.text,
            metadata={
                "source": doc.metadata.get("file_name", "Unknown"),
                "page_number": page_num  # Explicitly store it here
            }
        )
    )

# Before doing the semantic chunking. I am doing additional chunking. so, a chunk can't have more than 8192 tokens. which is maximum for model.
pre_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,       
    chunk_overlap=400,     # small overlap to preserve context at boundaries
)
pre_split_docs = pre_splitter.split_documents(lc_documents)
print(f"Pre-split into {len(pre_split_docs)} chunks before semantic chunking.")


chunks = SemanticChunker(
    embed_model,
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=95.0         # tune: lower = more chunks, higher = fewer
)


semantic_chunks = chunks.split_documents(pre_split_docs)

Pre-split into 298 chunks before semantic chunking.


In [6]:
print(f"Created {len(semantic_chunks)} semantic chunks using nomic-embed-text.")
print(f"Example Chunk Content:\n{semantic_chunks[0].page_content[:200]}...")

Created 1108 semantic chunks using nomic-embed-text.
Example Chunk Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zi...


Generating Embeddings and Using the Matryoshka Representation Learning(MRL) 

In [7]:
#We are using MRL to reduce the dimension of embeddings (To save space in the database and easy for search and to retreive the chunks from database)

Target_dim = 256

processed_chunks = []
print(f"Slicing vectors to {Target_dim} dimensions using MRL")

for chunk in semantic_chunks:
    full_vector = embed_model.embed_query(chunk.page_content)
    #we pulling the model from the Ollama to generate the embeddings
    sliced_vector = np.array(full_vector[:Target_dim])
    #MRL slicing. reducing from standard dimension to 256 dimension

    #we have to do renormalization for sliced vectors
    norm = np.linalg.norm(sliced_vector)
    if norm > 0:
        normalized_vector = sliced_vector / norm
    else:
        normalized_vector = sliced_vector

    #store the result with its metadata
    processed_chunks.append({
        "content": chunk.page_content,
        "metadata": chunk.metadata,
        "vector": normalized_vector.tolist()
    })


Slicing vectors to 256 dimensions using MRL


In [8]:
print(f"Ready for Weaviate: {len(processed_chunks)} vectors at {Target_dim} dims.")

Ready for Weaviate: 1108 vectors at 256 dims.


In [9]:
# # Connect to your local Weaviate instance
# client = weaviate.connect_to_local()

# try:
#     # Define the Collection (Table)
#     client.collections.create(
#         name="LlamaParse_MRL_Collection",
#         # We tell Weaviate: "I will provide the vectors, don't call an API"
#         vectorizer_config=Configure.Vectorizer.none(), 
#         properties=[
#             Property(name="content", data_type=DataType.TEXT),
#             Property(name="source", data_type=DataType.TEXT),
#         ]
#     )
    
#     collection = client.collections.get("LlamaParse_MRL_Collection")

#     # Efficient Batch Ingestion
#     with collection.batch.dynamic() as batch:
#         for item in processed_chunks:
#             batch.add_object(
#                 properties={
#                     "content": item["content"],
#                     "source": str(item["metadata"].get("file_path", "Unknown"))
#                 },
#                 vector=item["vector"] # Our 256-dim MRL vector
#             )
            
#     print("Successfully loaded all chunks into Weaviate!")

# finally:
#     client.close()

In [10]:
# 1. Connect to Weaviate
client = weaviate.connect_to_local()

try:
    collection_name = "LlamaParse_MRL_nomic"

    # Re-create the collection to ensure fresh metadata schema
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
    
    client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(), # Manual MRL vectors
        properties=[
            Property(name="content", data_type=DataType.TEXT),
            Property(name="source", data_type=DataType.TEXT),
            Property(name="page_number", data_type=DataType.TEXT), # Changed to TEXT to handle 'page_label'
        ]
    )
    
    collection = client.collections.get(collection_name)

    # 2. Batch Upload with Metadata Mapping
    print(f"Starting ingestion of {len(semantic_chunks)} chunks...")
    
    with collection.batch.dynamic() as batch:
        for i, chunk in enumerate(semantic_chunks):
            # A. Get full embedding from Ollama
            full_vector = embed_model.embed_query(chunk.page_content)
            
            # B. MRL Slice (256-dim) & Normalize
            sliced_vec = np.array(full_vector[:256])
            norm = np.linalg.norm(sliced_vec)
            final_vec = (sliced_vec / norm).tolist() if norm > 0 else sliced_vec.tolist()

            # C. Extract correct Page Label (Handles LlamaParse specific metadata)
            page_val = str(chunk.metadata.get("page_label", chunk.metadata.get("page_number", "Unknown")))

            batch.add_object(
                properties={
                    "content": chunk.page_content,
                    "source": str(chunk.metadata.get("file_name", "Unknown")),
                    "page_number": page_val
                },
                vector=final_vec
            )

    print(f"\n SUCCESS: Ingested {len(semantic_chunks)} chunks with Page Citations.")

finally:
    client.close()

c:\Users\jeeva\OneDrive\Desktop\C_projects\NLP\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Starting ingestion of 1108 chunks...

 SUCCESS: Ingested 1108 chunks with Page Citations.


User Query and Converting into Embedding and using MRL

In [ ]:
def user_query_embed():
    user_query = input("Type here")

    print("Generating embedding...")
    full_vector = embed_model.embed_query(user_query)

    #Applying the MRL
    target_dim = 256
    sliced_vector = np.array(full_vector[:target_dim])

    #Renormalization
    norm = np.linalg.norm(sliced_vector)

    if norm>0:
        normalized_query_vector = (sliced_vector/norm).tolist()
    else:
        normalized_query_vector = sliced_vector.tolist()

    print(f"User query converted into {len(normalized_query_vector)} dimensions")

    return user_query, normalized_query_vector

#Executing the step 1
query_text, query_vector = user_query_embed()


Generating embedding...
User query converted into 256 dimensions


Vector Chunk

In [ ]:
# def vector_search(query_vec):
#     top_k = int(input("Number of chunks should we retrive for context?"))

#     # connect to weaviate
#     client = weaviate.connect_to_local()

#     try:
#         #Accessing the MRL collection
#         collection = client.collections.get("LlamaParse_MRL_nomic")

#         print(f"Searching weaviate for top {top_k} matches...")\
#         # perform the Near-vector search
#         response  = collection.query.near_vector(
#             near_vector = query_vec,
#             limit = top_k,
#             return_properties = ["content", "page_number", "source"]
#         )
        
#         # Extract results
#         retrieved_results = []
#         for obj in response.objects:
#             retrieved_results.append(
#                 {
#                     "content": obj.properties["content"],
#                     "page": obj.properties["page_number"]
#                 }
#             )

#         print(f"Sucessfully retrieved {len(retrieved_results)} chunks.")

#         return retrieved_results
        
#     finally:
#         client.close()

# # Executing the step 2
# retrieved_chunks = vector_search(query_vector)

# # preview
# for i, chunk in enumerate(retrieved_chunks):
#     print(f"\n---Result {i+1} (page {chunk['page']})---")
#     print(f"{chunk['content'][:200]}....")


Vector search using the FAISS

In [21]:

def perform_faiss_search(query_vec, processed_chunks):
    # Ask user for the retrieval limit (k)
    k_limit = int(input("How many chunks should we retrieve for context? (e.g., 3, 5, 10): "))

    # Prepare the Index
    # We need to extract all vectors into a single 2D float32 numpy array
    all_vectors = np.array([item["vector"] for item in processed_chunks]).astype('float32')
    dimension = all_vectors.shape[1] # This should be 256
    
    # Create an IndexFlatIP (Inner Product) for normalized Cosine Similarity
    index = faiss.IndexFlatIP(dimension)
    index.add(all_vectors) # Add our MRL vectors to the index

    # Perform the Search
    # FAISS expects a 2D array for the query
    query_np = np.array([query_vec]).astype('float32')
    distances, indices = index.search(query_np, k_limit)

    # Extract Results using the indices
    retrieved_results = []
    print(f"FAISS searching for top {k_limit} matches...")
    
    for i in range(len(indices[0])):
        idx = indices[0][i]
        match = processed_chunks[idx]
        retrieved_results.append({
            "content": match["content"],
            "page": match["metadata"].get("page_number", "Unknown"),
            "score": float(distances[0][i])
        })
            
    print(f"FAISS successfully retrieved {len(retrieved_results)} chunks.")
    return retrieved_results

# Execute Step 2
retrieved_chunks = perform_faiss_search(query_vector, processed_chunks)

FAISS searching for top 25 matches...
FAISS successfully retrieved 25 chunks.
